# Module 1: Build a deep agent

> ~45 min · optional

You build the client-research agent one piece at a time with `deepagents`: built-in tools, your own tools, planning, subagents, memory, middleware, approvals, and skills. Every run is a trace in LangSmith, so you see what each piece adds. Two beta sections let the agent write code that drives its own tools and subagents. Modules 2–5 run a packaged version of this agent.

<img src="../images/deepAgentsDiag.png" style="width: auto; max-height: 420px; border-radius: 8px;">

## Setup
This cell loads your keys and the model, and sends every trace from this notebook to the **Local** environment of `client-research-agent`, so these runs stay apart from the traffic Engine reads in Module 2. Model calls go through the LangSmith LLM Gateway with `LANGSMITH_GATEWAY_API_KEY`.

> On a project-based workspace (self-hosted or older orgs), the cell sends traces to a separate `<your project>-build` project instead.

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# utils.config loads the repo's .env, so import it before anything reads env vars
from utils.config import AGENT_NAME, BUILD_ENVIRONMENT, PROJECT_NAME, TRACING_MODE, environment_project
from utils.models import MODEL_ID, make_model
from utils.notebook import print_files, trace_link

import langsmith as ls
from langgraph.checkpoint.memory import InMemorySaver
from langsmith import Client, uuid7

model = make_model()
client = Client()

if TRACING_MODE == "project":
    # Keeps build runs out of the project Engine watches
    build_project = client.create_project(f"{PROJECT_NAME}-build", upsert=True)
    trace_target = {"project_name": build_project.name}
    project_id, tracing_url = build_project.id, build_project.url
    destination_label = build_project.name
else:
    destination = environment_project(client, BUILD_ENVIRONMENT)
    trace_target = {"address": ls.agent_address(AGENT_NAME, BUILD_ENVIRONMENT)}
    project_id, tracing_url = destination.project_id, destination.url
    destination_label = f"{AGENT_NAME} / {BUILD_ENVIRONMENT}"


async def trace_url(run_id):
    """Return the LangSmith URL of the trace whose root run is run_id."""
    response = await client.runs.get_url(str(run_id), project_id=str(project_id), trace_id=str(run_id))
    return response.url or tracing_url


def tool_calls(result, name):
    """Return the main agent's calls to one tool, in order."""
    calls = []
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", []):
            if call["name"] == name:
                calls.append(call)
    return calls


print("Model:", MODEL_ID)
print("Tracing to:", destination_label)
trace_link(tracing_url, "Open this notebook's traces")

## 1. The harness
`create_deep_agent()` wraps a model in a harness: a virtual filesystem (`ls`, `read_file`, `write_file`, `edit_file`, `delete`, `glob`, `grep`) and a `task` tool backed by a built-in `general-purpose` subagent. Since deepagents 0.7 the default system prompt is empty and planning is opt-in, so this agent starts with tools and no instructions.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(model=model, checkpointer=InMemorySaver())

HAIKU_REQUEST = "Write a haiku about markets to /haiku.txt, then read it back to me."
thread = {"thread_id": str(uuid7())}
run_id = uuid7()
with ls.tracing_context(**trace_target):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": HAIKU_REQUEST}]},
        config={"configurable": thread, "run_id": run_id},
    )

print("Reply:", result["messages"][-1].text)
print_files(result.get("files", {}))
trace_link(await trace_url(run_id))

Open the trace: the model calls `write_file` and then `read_file`, tools you never defined, and the file lives in agent state, not on disk.

<img src="../images/TODO-01-local-environment.png" style="width: auto; max-height: 380px; border-radius: 8px;">

*The haiku trace under **Tracing** → `client-research-agent` → **Local**.*

In [ ]:
run_id = uuid7()
with ls.tracing_context(**trace_target):
    same_thread = agent.invoke(
        {"messages": [{"role": "user", "content": "Read /haiku.txt"}]},
        config={"configurable": thread, "run_id": run_id},
    )
    new_thread = agent.invoke(
        {"messages": [{"role": "user", "content": "List all files with ls /"}]},
        config={"configurable": {"thread_id": str(uuid7())}},
    )

print("Same thread:", same_thread["messages"][-1].text)
print("New thread:", new_thread["messages"][-1].text)
trace_link(await trace_url(run_id), "View the same-thread trace")

The default **StateBackend** keeps files per thread through the checkpointer, so the new thread starts with `No files found`. Section 5 makes some paths outlast the thread.

## 2. Your own tools and prompt
Tools you pass sit next to the built-in ones, and the `system_prompt` you pass is the agent's whole authored prompt. `web_search` is defined here; its body calls `utils/search.py`, which retries Tavily and falls back to a canned result.

In [ ]:
from langchain_core.tools import tool
from utils.search import resilient_tavily_search


@tool(parse_docstring=True)
def web_search(query: str) -> str:
    """Search the web for company news, market data, or general information.

    Args:
        query: Search query to execute.
    """
    return resilient_tavily_search(query, max_retries=2)


SYSTEM_PROMPT = (
    "You are a client research assistant for a wealth management firm. "
    "Answer in plain language and cite the sources you used."
)
agent = create_deep_agent(model=model, tools=[web_search], system_prompt=SYSTEM_PROMPT, checkpointer=InMemorySaver())

SUMMARY_REQUEST = "Summarize recent news on Apple (AAPL) in one paragraph and write it to /summary.md. Search at most once."
run_id = uuid7()
with ls.tracing_context(**trace_target):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": SUMMARY_REQUEST}]},
        config={"configurable": {"thread_id": str(uuid7())}, "run_id": run_id},
    )

print("Reply:", result["messages"][-1].text)
print_files(result.get("files", {}))
trace_link(await trace_url(run_id))

In the trace, `web_search` sits beside `write_file`. Open the first model run: its system prompt starts with `SYSTEM_PROMPT`.

## 3. Planning with to-dos
**TodoListMiddleware** adds a `write_todos` tool that keeps a task list in agent state. It is opt-in since deepagents 0.7, and long multi-step tasks and smaller models benefit most.

In [ ]:
from langchain.agents.middleware import TodoListMiddleware

agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt=SYSTEM_PROMPT,
    middleware=[TodoListMiddleware()],
    checkpointer=InMemorySaver(),
)

COMPARE_REQUEST = (
    "Plan your steps first, then compare the latest quarterly results of Apple and Microsoft "
    "and write the comparison to /compare.md. Search at most twice."
)
run_id = uuid7()
with ls.tracing_context(**trace_target):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": COMPARE_REQUEST}]},
        config={"configurable": {"thread_id": str(uuid7())}, "run_id": run_id},
    )

print("To-dos:")
for todo in result.get("todos", []):
    print(f"  {todo['status']:<12} {todo['content']}")
trace_link(await trace_url(run_id))

Each `write_todos` call in the trace is a plan update: steps move from `pending` to `in_progress` to `completed`.

## 4. Subagents
A **subagent** runs in its own context: the main agent calls it with `task()` and sees only its final answer. The name and description tell the main agent when to use it, and a `general-purpose` subagent is always there unless you replace it.

<img src="../images/deepAgentSubagents.png" style="width: auto; max-height: 380px; border-radius: 8px;">

In [ ]:
from datetime import date

research_subagent = {
    "name": "research-agent",
    "description": "Researches one company or market topic on the web. Give it one topic at a time.",
    "system_prompt": (
        f"You are a research assistant for a wealth management firm. Today is {date.today():%Y-%m-%d}. "
        "Use web_search at most 2 times, then answer with cited sources."
    ),
    "tools": [web_search],
}
agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt=SYSTEM_PROMPT,
    subagents=[research_subagent],
    checkpointer=InMemorySaver(),
)

NEWS_REQUEST = "Lightly research recent news on NVIDIA. Delegate the research to the research-agent."
run_id = uuid7()
with ls.tracing_context(**trace_target):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": NEWS_REQUEST}]},
        config={"configurable": {"thread_id": str(uuid7())}, "run_id": run_id},
    )

print("task calls:", len(tool_calls(result, "task")))
print("Reply:", result["messages"][-1].text[:1500])
trace_link(await trace_url(run_id))

In the trace, the `task` run holds a full `research-agent` run with its own model calls and searches, while the main agent's context holds only its last message.

## 5. Memory across threads
**CompositeBackend** routes paths to different backends: here `/memories/` goes to a **StoreBackend**, which outlasts the thread, and everything else stays in state. Since deepagents 0.7 you pass backend instances and give `StoreBackend` an explicit `namespace`.

<img src="../images/deepAgentBackends.png" style="width: auto; max-height: 380px; border-radius: 8px;">

In [ ]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
backend = CompositeBackend(
    default=StateBackend(),
    routes={"/memories/": StoreBackend(store=store, namespace=lambda runtime: ("memories", "shared"))},
)
MEMORY_PROMPT = (
    f"{SYSTEM_PROMPT} Save client facts under /memories/clients/. "
    "Check /memories/ before you answer a question about a client."
)
agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt=MEMORY_PROMPT,
    backend=backend,
    store=store,
    checkpointer=InMemorySaver(),
)

REMEMBER_REQUEST = (
    "Remember that Smith Family Office's risk profile is aggressive growth. "
    "Save it to /memories/clients/smith-family-office.md"
)
run_id = uuid7()
with ls.tracing_context(**trace_target):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": REMEMBER_REQUEST}]},
        config={"configurable": {"thread_id": str(uuid7())}, "run_id": run_id},
    )

print("Thread 1:", result["messages"][-1].text)
print("Saved in the store:", [item.key for item in store.search(("memories", "shared"))])

In [ ]:
RECALL_REQUEST = "What is Smith Family Office's risk profile? Check /memories/ first."
run_id = uuid7()
with ls.tracing_context(**trace_target):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": RECALL_REQUEST}]},
        config={"configurable": {"thread_id": str(uuid7())}, "run_id": run_id},
    )

print("Thread 2:", result["messages"][-1].text)
trace_link(await trace_url(run_id))

Thread 2's trace starts with `ls` or `read_file` under `/memories/`: the answer comes from the store, not from the conversation.

## 6. Middleware
Middleware wraps every model call (`wrap_model_call`) or tool call (`wrap_tool_call`), so you add rules or logging without touching the agent's tools or prompt. The harness itself uses middleware to offload large tool results to the filesystem and to summarize older history as the context window fills.

<img src="../images/deepAgentMiddleware.png" style="width: auto; max-height: 380px; border-radius: 8px;">

In [ ]:
from datetime import datetime

from langchain.agents.middleware import wrap_model_call, wrap_tool_call
from langchain_core.messages import SystemMessage

COMPLIANCE_RULES = (
    "## Compliance rules\n"
    "- Never include account numbers or passwords in a response.\n"
    "- Cite a source for every market fact.\n"
    "- Flag any request for non-public client information."
)
audit_log = []


@wrap_model_call
def compliance_rules(request, handler):
    blocks = list(request.system_message.content_blocks) if request.system_message else []
    blocks.append({"type": "text", "text": f"\n\n{COMPLIANCE_RULES}"})
    return handler(request.override(system_message=SystemMessage(content_blocks=blocks)))


@wrap_tool_call
def audit_trail(request, handler):
    result = handler(request)
    status = getattr(result, "status", "success")  # a Command result carries no status
    audit_log.append({"tool": request.tool_call["name"], "status": status, "at": datetime.now().strftime("%H:%M:%S")})
    return result


print("Middleware ready:", compliance_rules.name, audit_trail.name)

In [ ]:
audit_log.clear()
agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt=SYSTEM_PROMPT,
    middleware=[compliance_rules, audit_trail],
    checkpointer=InMemorySaver(),
)

EARNINGS_REQUEST = "Summarize Microsoft's latest earnings and write the summary to /summary.md. Search at most once."
run_id = uuid7()
with ls.tracing_context(**trace_target):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": EARNINGS_REQUEST}]},
        config={"configurable": {"thread_id": str(uuid7())}, "run_id": run_id},
    )

print("Reply:", result["messages"][-1].text)
print("Audit log entries:", len(audit_log))
for entry in audit_log:
    print(f"  {entry['at']}  {entry['tool']:<20} {entry['status']}")
trace_link(await trace_url(run_id))

The model run in the trace shows the compliance rules at the end of its system prompt, and the audit log has one row per tool call in the trace.

## 7. Human approval
`interrupt_on` pauses the agent before the tools you list, and you approve, edit, or reject each call. Here it gates `write_file`, which overwrites existing files since deepagents 0.7, and `delete`, which removes directories recursively.

<img src="../images/deepAgentHITL.png" style="width: auto; max-height: 380px; border-radius: 8px;">

In [ ]:
agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt=SYSTEM_PROMPT,
    interrupt_on={"write_file": True, "delete": True},
    checkpointer=InMemorySaver(),
)

NOTE_REQUEST = "Write a one-paragraph note on NVIDIA to /nvda_note.md. Search at most once."
config = {"configurable": {"thread_id": str(uuid7())}}
run_id = uuid7()
with ls.tracing_context(**trace_target):
    result = agent.invoke({"messages": [{"role": "user", "content": NOTE_REQUEST}]}, config={**config, "run_id": run_id})

for action in result["__interrupt__"][0].value["action_requests"]:
    print("Paused:", action["name"], action["args"].get("file_path"))
trace_link(await trace_url(run_id))

In [ ]:
from langgraph.types import Command

actions = result["__interrupt__"][0].value["action_requests"]
decisions = [{"type": "approve"} for _ in actions]
run_id = uuid7()
with ls.tracing_context(**trace_target):
    result = agent.invoke(Command(resume={"decisions": decisions}), config={**config, "run_id": run_id})

print("Approved:", len(decisions))
print("Reply:", result["messages"][-1].text)
print_files(result.get("files", {}))
trace_link(await trace_url(run_id))

The thread has two traces: the first ends at the pause, and the second resumes from the checkpoint and runs `write_file`. Open **Threads** in the Local environment to see both together.

## 8. AGENTS.md and skills
**Memory** files such as `AGENTS.md` load into the system prompt on every call, while **skills** load only their name and description up front, and the agent reads the full `SKILL.md` when a task matches. With the default StateBackend you pass both as files in the input.

In [ ]:
from deepagents.backends.utils import create_file_data

AGENTS_MD = """# Client Research Assistant

You research clients and markets for a wealth management firm and write pre-meeting briefs.

## Workflow
1. If a client is named, call get_client_profile first.
2. Delegate company and market research to the research-agent with task().
3. Write the brief to /client_brief.md.
4. Save key takeaways to /memories/client_notes.md.

## Rules
- Cite sources as [1], [2], [3] and end with a Sources list.
- Check /skills/ when asked for a specific format, such as a client brief.
"""

CLIENT_BRIEF_SKILL = """---
name: client-brief
description: Write a structured pre-meeting brief for a relationship manager. Use this skill when asked to prep a client meeting, summarize a client's situation, or produce a meeting prep document.
---

# Client Brief Skill

## Format
- **Header**: Client name, AUM, last meeting date
- **Portfolio Snapshot**: Top holdings with weights, risk profile (1-2 sentences)
- **Holdings Highlights**: 2-3 sentences per major holding covering recent news, key drivers, current risks
- **Market Context**: Brief paragraph on broader market themes relevant to the portfolio
- **Talking Points**: 3-5 bulleted prompts the RM can raise in the meeting
- **Sources**: Numbered list of citations referenced above

## Tone
- Concise and factual; lead with what changed since the last meeting; flag risks plainly.

## Length
- 400-600 words.
"""

seed_files = {
    "/AGENTS.md": create_file_data(AGENTS_MD),
    "/skills/client-brief/SKILL.md": create_file_data(CLIENT_BRIEF_SKILL),
}
print("Seed files:", list(seed_files))

In [ ]:
agent = create_deep_agent(
    model=model,
    tools=[web_search],
    subagents=[research_subagent],
    middleware=[TodoListMiddleware()],
    memory=["/AGENTS.md"],
    skills=["/skills/"],
    checkpointer=InMemorySaver(),
)

BRIEF_REQUEST = (
    "Prep a client brief for a family office that holds NVDA 12%, GOOGL 10% and BRK.B 8% "
    "and has an aggressive growth risk profile. Research one holding. Search at most once."
)
run_id = uuid7()
with ls.tracing_context(**trace_target):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": BRIEF_REQUEST}], "files": seed_files},
        config={"configurable": {"thread_id": str(uuid7())}, "run_id": run_id},
    )

print("Files read:", [call["args"].get("file_path") for call in tool_calls(result, "read_file")])
print("Reply:", result["messages"][-1].text[:1500])
trace_link(await trace_url(run_id))

`Files read` lists `/skills/client-brief/SKILL.md`: the agent pulled the skill in only after the request matched its description. In the trace, that `read_file` call comes before the brief is written.